In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import os
import random
import bisect
import time
from os.path import join, exists
from tqdm.notebook import tqdm

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


Device: cuda
GPU: Tesla T4


In [ ]:
DATA_DIR = '/kaggle/input/datasets/tungngo1207/500-masque-new'
CHECKPOINT_DIR = '/kaggle/working/netclr_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

SEQ_LENGTH = 10000       # NetCLR dung 10000 direction values
BATCH_SIZE = 128
TEMPERATURE = 0.5        # SimCLR temperature
PRETRAIN_EPOCHS = 50    # SimCLR pretraining
FINETUNE_EPOCHS = 50    # Classification finetuning
FINETUNE_LR = 0.0001
PRETRAIN_LR = 0.0003
TEST_SIZE = 0.1          # Giong Var-CNN chia 10% data test, 90% train

print(f'Data: {DATA_DIR}')
print(f'Seq length: {SEQ_LENGTH}')
print(f'Pretrain: {PRETRAIN_EPOCHS} epochs, LR={PRETRAIN_LR}, temp={TEMPERATURE}')
print(f'Finetune: {FINETUNE_EPOCHS} epochs, LR={FINETUNE_LR}')


Data: /kaggle/input/datasets/tungngo1207/500-masque-new
Seq length: 10000
Pretrain: 50 epochs, LR=0.0003, temp=0.5
Finetune: 50 epochs, LR=0.0001


In [ ]:
# =========================================================
# LOAD DATA 
# =========================================================

print('Loading data...')
classes = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(join(DATA_DIR, d))])
print(f'Found {len(classes)} classes')

all_x = []
all_y = []

for cls_idx, cls_name in enumerate(tqdm(classes, desc='Loading')):
    cls_dir = join(DATA_DIR, cls_name)
    files = sorted([f for f in os.listdir(cls_dir) if f.endswith('.csv')])
    
    for fname in files:
        try:
            df = pd.read_csv(join(cls_dir, fname), sep=';')
            if len(df) < 100:
                continue
            
            # Direction: 0 -> -1 (outgoing), 1 -> +1 (incoming)
            dirs = df['direction'].values
            dirs = np.where(dirs == 0, -1, 1).astype(np.int8)
            
            # Pad/truncate to SEQ_LENGTH
            vec = np.zeros(SEQ_LENGTH, dtype=np.int8)
            limit = min(len(dirs), SEQ_LENGTH)
            vec[:limit] = dirs[:limit]
            
            all_x.append(vec)
            all_y.append(cls_idx)
        except Exception:
            continue

x_data = np.array(all_x, dtype=np.float32)
y_data = np.array(all_y, dtype=np.int64)
num_classes = len(classes)

print(f'\nLoaded: {len(x_data)} samples, {num_classes} classes')
print(f'x shape: {x_data.shape}')

# Train/Test split 
x_train, x_test, y_train, y_test = train_test_split(
    x_data, y_data, test_size=TEST_SIZE, random_state=42, stratify=y_data)

print(f'Train: {x_train.shape}, Test: {x_test.shape}')

# Class distribution
for i, cls in enumerate(classes):
    n_train = np.sum(y_train == i)
    n_test = np.sum(y_test == i)
    print(f'  {cls:30s}: train={n_train}, test={n_test}')


Loading data...
Found 500 classes


Loading:   0%|          | 0/500 [00:00<?, ?it/s]


Loaded: 247035 samples, 500 classes
x shape: (247035, 10000)
Train: (222331, 10000), Test: (24704, 10000)
  24tv_ua                       : train=448, test=50
  2beeg_me                      : train=448, test=50
  3gpking_name                  : train=448, test=50
  a1_bluesystem_me              : train=449, test=50
  aagmaal_run                   : train=449, test=50
  about_google                  : train=449, test=50
  acronis_com                   : train=441, test=49
  activebeat_com                : train=448, test=50
  aeolservice_es                : train=448, test=50
  albumaty_com                  : train=448, test=50
  amarmatka_co                  : train=447, test=49
  anchor_fm                     : train=448, test=50
  aniwave_to                    : train=445, test=49
  anysex_com                    : train=450, test=50
  apnews_com                    : train=449, test=50
  appspot_com                   : train=421, test=47
  apptimize_com                 : train=448, 

In [4]:
# =========================================================
# DFNet MODEL (Deep Fingerprinting backbone)
# =========================================================

class DFNet(nn.Module):
    def __init__(self, out_dim):
        super(DFNet, self).__init__()
        kernel_size = 8
        channels = [1, 32, 64, 128, 256]
        conv_stride = 1
        pool_stride = 4
        pool_size = 8
        
        self.conv1 = nn.Conv1d(1, 32, kernel_size, stride = conv_stride)
        self.conv1_1 = nn.Conv1d(32, 32, kernel_size, stride = conv_stride)
        
        self.conv2 = nn.Conv1d(32, 64, kernel_size, stride = conv_stride)
        self.conv2_2 = nn.Conv1d(64, 64, kernel_size, stride = conv_stride)
       
        self.conv3 = nn.Conv1d(64, 128, kernel_size, stride = conv_stride)
        self.conv3_3 = nn.Conv1d(128, 128, kernel_size, stride = conv_stride)
       
        self.conv4 = nn.Conv1d(128, 256, kernel_size, stride = conv_stride)
        self.conv4_4 = nn.Conv1d(256, 256, kernel_size, stride = conv_stride)
       
        
        self.batch_norm1 = nn.BatchNorm1d(32)
        self.batch_norm2 = nn.BatchNorm1d(64)
        self.batch_norm3 = nn.BatchNorm1d(128)
        self.batch_norm4 = nn.BatchNorm1d(256)
        
        self.max_pool_1 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
        self.max_pool_2 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
        self.max_pool_3 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
        self.max_pool_4 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
        
        self.dropout1 = nn.Dropout(p=0.1)
        self.dropout2 = nn.Dropout(p=0.1)
        self.dropout3 = nn.Dropout(p=0.1)
        self.dropout4 = nn.Dropout(p=0.1)

        
        self.fc = nn.Linear(10240, out_dim)

        
    def weight_init(self):
        for n, m in self.named_modules():
            if isinstance(m, nn.Linear) or isinstance(m, nn.Conv1d):
#                 m.weight.data.xavier_uniform_()
                # print (n)
                torch.nn.init.xavier_uniform(m.weight)
                m.bias.data.zero_()
            
        
    def forward(self, inp):
        x = inp
        # ==== first block ====
        x = F.pad(x, (3,4))
        x = F.elu((self.conv1(x)))
        x = F.pad(x, (3,4))
        x = F.elu(self.batch_norm1(self.conv1_1(x)))
#         x = F.elu(self.conv1_1(x))
        x = F.pad(x, (3, 4))
        x = self.max_pool_1(x)
        x = self.dropout1(x)
        
        # ==== second block ====
        x = F.pad(x, (3,4))
        x = F.relu((self.conv2(x)))
        x = F.pad(x, (3,4))
        x = F.relu(self.batch_norm2(self.conv2_2(x)))
#         x = F.relu(self.conv2_2(x))
        x = F.pad(x, (3,4))
        x = self.max_pool_2(x)
        x = self.dropout2(x)
        
        # ==== third block ====
        x = F.pad(x, (3,4))
        x = F.relu((self.conv3(x)))
        x = F.pad(x, (3,4))
        x = F.relu(self.batch_norm3(self.conv3_3(x)))
#         x = F.relu(self.conv3_3(x))
        x = F.pad(x, (3,4))
        x = self.max_pool_3(x)
        x = self.dropout3(x)
        
        # ==== fourth block ====
        x = F.pad(x, (3,4))
        x = F.relu((self.conv4(x)))
        x = F.pad(x, (3,4))
        x = F.relu(self.batch_norm4(self.conv4_4(x)))
#         x = F.relu(self.conv4_4(x))
        x = F.pad(x, (3,4))
        x = self.max_pool_4(x)
        x = self.dropout4(x)

                
        x = x.view(x.size(0), -1)
        
#         x = self.projection(x)

        x = self.fc(x)
                
        return x    
        


class DFsimCLR(nn.Module):
    def __init__(self, df, out_dim):
        super(DFsimCLR, self).__init__()
        
        self.backbone = df
        self.backbone.weight_init()
        dim_mlp = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(dim_mlp, dim_mlp),
            nn.BatchNorm1d(dim_mlp),
            nn.ReLU(),
            nn.Linear(dim_mlp, out_dim)
        )
        
    def forward(self, inp):
        out = self.backbone(inp)
        return out

In [5]:
# =========================================================
# BURST ANALYSIS + AUGMENTOR (from NetCLR paper)
# =========================================================

# Compute burst CDF from training data
print('Computing burst statistics...')

def find_bursts(x):
    if len(x) == 0 or x[0] == 0:
        return []
    direction = x[0]
    bursts = []
    start = 0
    temp_burst = x[0]
    for i in range(1, len(x)):
        if x[i] == 0.0:
            break
        elif x[i] == direction:
            temp_burst += x[i]
        else:
            bursts.append((start, i, temp_burst))
            start = i
            temp_burst = x[i]
            direction *= -1
    return bursts

outgoing_burst_sizes = []
sample_idx = np.random.choice(len(x_train), size=min(650, len(x_train)), replace=False)
for idx in sample_idx:
    bursts = find_bursts(x_train[idx])
    outgoing_burst_sizes += [b[2] for b in bursts if b[2] > 0]

if len(outgoing_burst_sizes) == 0:
    outgoing_burst_sizes = [1]
max_outgoing_burst_size = int(max(outgoing_burst_sizes))  

count, bins = np.histogram(outgoing_burst_sizes, bins=max(max_outgoing_burst_size - 1, 1))
PDF = count / np.sum(count)
OUTGOING_BURST_SIZE_CDF = np.zeros_like(bins)
OUTGOING_BURST_SIZE_CDF[1:] = np.cumsum(PDF)

print(f'Max outgoing burst: {max_outgoing_burst_size}')
print(f'Burst CDF computed from {len(outgoing_burst_sizes)} bursts')

# Augmentor class (from NetCLR)
class Augmentor():
    def __init__(self):
        methods = {
            'merge downstream burst',
            'change downstream burst sizes',
            'merge downstream and upstream bursts',
            'add upstream bursts',
            'remove upstrean bursts',
            'divide bursts'
        }
        
        
        self.large_burst_threshold = 10
        
        # changing the content
        self.upsample_rate = 1.0
        self.downsample_rate = 0.5
        
        # merging bursts
        self.num_bursts_to_merge = 5
        self.merge_burst_rate = 0.1
        
        # add incoming bursts
        self.add_outgoing_burst_rate = 0.3
        self.outgoing_burst_sizes = list(range(max_outgoing_burst_size))
        
        # shift
        self.shift_param = 10
        
        
        
    def find_bursts(self, x):
        direction = x[0]
        bursts = []
        start = 0
        temp_burst = x[0]
        for i in range(1, len(x)):
            if x[i] == 0.0:
                break

            elif x[i] == direction:
                temp_burst += x[i]

            else:
                # if temp_burst <= -10 or temp_burst > 0:
                bursts.append((start, i, temp_burst))
                start = i
                temp_burst = x[i]
                direction *= -1

        return bursts
        
        
    # representing the change of contents of a website
    def increase_incoming_bursts(self, burst_sizes):
        out = []
        for i, size in enumerate(burst_sizes):
            if size <= -self.large_burst_threshold:
                up_sample_rate = random.random()*self.upsample_rate
                new_size = int(size * (1+up_sample_rate))
                out.append(new_size)
            else:
                out.append(size)
                
        return out
        
        
    def decrease_incoming_bursts(self, burst_sizes):
        out = []
        for i, size in enumerate(burst_sizes):
            if size <= -self.large_burst_threshold:
                up_sample_rate = random.random()*self.downsample_rate
                new_size = int(size * (1-up_sample_rate))
                out.append(new_size)
            else:
                out.append(size)
                
        return out
        
        
    def change_content(self, burst_sizes):  # receives burst_sizes, not trace
        # DON'T call find_bursts again - burst_sizes already passed in
        
        # Count non-zero packets to determine trace "length"
        total_packets = sum(abs(s) for s in burst_sizes)
        
        if total_packets < 1000:
            new_burst_sizes = self.increase_incoming_bursts(burst_sizes)
        elif total_packets > 4000:
            new_burst_sizes = self.decrease_incoming_bursts(burst_sizes)
        else:
            p = random.random()
            if p >= 0.5:
                new_burst_sizes = self.increase_incoming_bursts(burst_sizes)
            else:
                new_burst_sizes = self.decrease_incoming_bursts(burst_sizes)
        
        return new_burst_sizes
    
    
    def merge_incoming_bursts(self, burst_sizes):
        
        out = []
        
        # skipping first 20 cells
        i = 0
        num_cells = 0
        while i < len(burst_sizes) and num_cells < 20:
            num_cells += abs(burst_sizes[i])
            out.append(burst_sizes[i])
            i += 1
            
        
        while i < len(burst_sizes) - self.num_bursts_to_merge:
            prob = random.random()
            
            # ignore outgoing bursts
            if burst_sizes[i] > 0:
                out.append(burst_sizes[i])
                i+= 1
                continue
            
            if prob < self.merge_burst_rate:
                num_merges = random.randint(2, self.num_bursts_to_merge)
                merged_size = 0
                
                # merging the incoming bursts
                while i < len(burst_sizes) and num_merges > 0:
                    if burst_sizes[i] < 0:
                        merged_size += burst_sizes[i]
                        num_merges -= 1
                    i += 1     
                out.append(merged_size)
                    
            else:
                out.append(burst_sizes[i])
                i += 1
                
        return out
    
    
    def add_outgoing_burst(self, burst_sizes):
        
        out = []
        
        i = 0
        num_cells = 0
        while i < len(burst_sizes) and num_cells < 20:
            num_cells += abs(burst_sizes[i])
            out.append(burst_sizes[i])
            i += 1
            
        
        for size in burst_sizes[i:]:
            if size > -10 :
                out.append(size)
                continue
            
            prob = random.random()
            
            if prob < self.add_outgoing_burst_rate:
                
                index = len(outgoing_burst_sizes)
                while index >= len(outgoing_burst_sizes):
                    outgoing_burst_prob = random.random()
                    index = bisect.bisect_left(OUTGOING_BURST_SIZE_CDF, outgoing_burst_prob)
                    
                outgoing_burst_size = self.outgoing_burst_sizes[index]
                abs_size = int(abs(size))
                if abs_size > 6:  # đảm bảo có khoảng hợp lệ: randint(3, abs_size-3)
                    divide_place = random.randint(3, abs_size - 3)
                else:
                    divide_place = abs_size // 2  # fallback an toàn
                
                out += [-divide_place, outgoing_burst_size, -(abs(size) - divide_place)]
                
            else:
                out.append(size)
                
        return out
                
        
    def create_trace_from_burst_sizes(self, burst_sizes):
        out = []
        
        for size in burst_sizes:
            val = 1 if size > 0 else -1
            
            out += [val]*(int(abs(size)))
            
        if len(out) < 10000:
            out += [0]*(10000 - len(out))
            
        return np.array(out)[:10000]
    
    def shift(self, x):
        pad = np.random.randint(0, 2, size = (self.shift_param, ))
        pad = 2*pad-1
        zpad = np.zeros_like(pad)
        
        shift_val = np.random.randint(-self.shift_param, self.shift_param+1, 1)[0]
        shifted = np.concatenate((x, zpad, pad), axis=-1)
        shifted = np.roll(shifted, shift_val, axis=-1)
        shifted = shifted[:10000]
        
        return shifted
        
    
    def augment(self, trace):
        bursts = self.find_bursts(trace)
        burst_sizes = [x[2] for x in bursts]
        
        if len(burst_sizes) == 0:  # guard against empty traces
            return self.shift(trace.copy())
        
        mapping = {
            0: self.change_content,
            1: self.merge_incoming_bursts,
            2: self.add_outgoing_burst
        }
        
        aug_method = mapping[random.randint(0, len(mapping)-1)]
        augmented_sizes = aug_method(burst_sizes)
        augmented_trace = self.create_trace_from_burst_sizes(augmented_sizes)
        
        return self.shift(augmented_trace)


print('Augmentor ready!')


Computing burst statistics...
Max outgoing burst: 236
Burst CDF computed from 701957 bursts
Augmentor ready!


In [6]:
# =========================================================
# DATASET + UTILS
# =========================================================

class TrainData(Dataset):
    def __init__(self, x_train, y_train, augmentor, n_views):
        self.x = x_train
        self.y = y_train
        self.augmentor = augmentor
        self.n_views = n_views
    
    def __getitem__(self, index):
        return [self.augmentor.augment(self.x[index]) for i in range(self.n_views)], self.y[index]
    
    def __len__(self):
        return len(self.x)

class TestData(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __getitem__(self, index):
        return self.x[index], self.y[index]
    def __len__(self):
        return len(self.x)

def accuracy(output, target, topk=(1,)):
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)
        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))
        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
            res.append(correct_k.mul_(100.0 / batch_size))
        return res

print('Dataset classes ready!')


Dataset classes ready!


In [ ]:
# =========================================================
# SIMCLR PRETRAINING (NetCLR)
# =========================================================

class NetCLR(object):
    def __init__(self, model, optimizer, scheduler, temperature, batch_size, device):
        self.model = model
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.temperature = temperature
        self.batch_size = batch_size
        self.device = device
        self.criterion = torch.nn.CrossEntropyLoss().to(device)
    
    def info_nce_loss(self, features):
        labels = torch.cat([torch.arange(self.batch_size) for _ in range(2)], dim=0)
        labels = (labels.unsqueeze(0) == labels.unsqueeze(1)).float().to(self.device)
        features = F.normalize(features, dim=1)
        sim_matrix = torch.matmul(features, features.T)
        mask = torch.eye(labels.shape[0], dtype=torch.bool).to(self.device)
        labels = labels[~mask].view(labels.shape[0], -1)
        sim_matrix = sim_matrix[~mask].view(sim_matrix.shape[0], -1)
        positives = sim_matrix[labels.bool()].view(labels.shape[0], -1)
        negatives = sim_matrix[~labels.bool()].view(sim_matrix.shape[0], -1)
        logits = torch.cat([positives, negatives], dim=1) / self.temperature
        labels = torch.zeros(logits.shape[0], dtype=torch.long).to(self.device)
        return logits, labels
    
    def train(self, train_loader, start_epoch, num_epochs):
        scaler = GradScaler(enabled=True)
        print(f'Starting SimCLR pretraining (epoch {start_epoch+1} to {num_epochs})...')
        
        for epoch in range(start_epoch, num_epochs):
            epoch_loss = []
            pbar = tqdm(train_loader, desc=f'Pretrain {epoch+1}/{num_epochs}', leave=True)
            
            for data, _ in pbar:
                self.model.train()
                data = torch.cat(data, dim=0)
                data = data.view(data.size(0), 1, data.size(1)).float().to(self.device)
                
                with autocast(enabled=True):
                    features = self.model(data)
                    logits, labels = self.info_nce_loss(features)
                    loss = self.criterion(logits, labels)
                
                self.optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(self.optimizer)
                scaler.update()
                
                epoch_loss.append(loss.item())
                pbar.set_postfix(loss=f'{loss.item():.4f}')
            
            if epoch >= 10:
                self.scheduler.step()
            
            avg_loss = np.mean(epoch_loss)
            print(f'  Epoch {epoch+1} avg loss: {avg_loss:.4f}')
            
            # Checkpoint moi 10 epochs
            if (epoch + 1) % 10 == 0:
                ckpt_path = join(CHECKPOINT_DIR, f'netclr_epoch{epoch+1}.pth')
                torch.save(self.model.state_dict(), ckpt_path)
                print(f'  Saved: {ckpt_path}')

# --- Build model ---
print('Building NetCLR model...')
augmentor = Augmentor()

train_dataset = TrainData(x_train, y_train, augmentor, 2)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                          drop_last=True, num_workers=2, pin_memory=True)

df = DFNet(out_dim=512)
model = DFsimCLR(df, out_dim=128).to(device)
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')

# --- Check checkpoint ---
import glob
ckpts = glob.glob('/kaggle/input/datasets/lampdp/resume2/netclr_checkpoints/*.pth', recursive=True)
local_ckpts = glob.glob(join(CHECKPOINT_DIR, '*.pth'))
all_ckpts = ckpts + local_ckpts
print(f'Found checkpoints: {all_ckpts}')

# Tim epoch cao nhat hoac final
start_epoch = 0
final_found = False

if all_ckpts:
    final = [c for c in all_ckpts if 'final' in c]
    if final:
        print(f'Loading final checkpoint: {final[0]}')
        model.load_state_dict(torch.load(final[0], map_location=device))
        final_found = True
        print('SKIPPING PRETRAINING (final checkpoint found)')
    else:
        # Tim epoch cao nhat
        epoch_ckpts = []
        for c in all_ckpts:
            fname = os.path.basename(c)
            if 'epoch' in fname:
                try:
                    ep = int(fname.replace('netclr_epoch', '').replace('.pth', ''))
                    epoch_ckpts.append((ep, c))
                except:
                    pass
        if epoch_ckpts:
            epoch_ckpts.sort(key=lambda x: x[0])
            best_ep, best_path = epoch_ckpts[-1]
            print(f'Loading checkpoint epoch {best_ep}: {best_path}')
            model.load_state_dict(torch.load(best_path, map_location=device))
            start_epoch = best_ep
            print(f'Will resume from epoch {start_epoch+1}')

if not final_found:
    optimizer = torch.optim.Adam(model.parameters(), lr=PRETRAIN_LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=len(train_loader), eta_min=0, last_epoch=-1)
    netclr = NetCLR(model, optimizer, scheduler, TEMPERATURE, BATCH_SIZE, device)
    
    if start_epoch >= PRETRAIN_EPOCHS:
        print(f'Already completed {start_epoch} epochs, skipping pretrain')
    else:
        start = time.time()
        netclr.train(train_loader, start_epoch, PRETRAIN_EPOCHS)
        print(f'\nPretraining done in {time.time()-start:.0f}s')

# Save final
final_path = join(CHECKPOINT_DIR, 'netclr_final.pth')
torch.save(model.state_dict(), final_path)
print(f'Final model saved: {final_path}')


Building NetCLR model...
Model params: 107,241,728
Found checkpoints: ['/kaggle/input/datasets/lampdp/resume2/netclr_checkpoints/netclr_final.pth', '/kaggle/input/datasets/lampdp/resume2/netclr_checkpoints/netclr_epoch40.pth', '/kaggle/input/datasets/lampdp/resume2/netclr_checkpoints/netclr_epoch50.pth', '/kaggle/input/datasets/lampdp/resume2/netclr_checkpoints/netclr_epoch30.pth']
Loading final checkpoint: /kaggle/input/datasets/lampdp/resume2/netclr_checkpoints/netclr_final.pth
SKIPPING PRETRAINING (final checkpoint found)
Final model saved: /kaggle/working/netclr_checkpoints/netclr_final.pth


In [8]:
# =========================================================
# FINE-TUNING + EVALUATION
# =========================================================

print('\n' + '='*60)
print('FINE-TUNING')
print('='*60)

# Load pretrained backbone, thay fc head
model_ft = DFNet(out_dim=num_classes).to(device)

# Load pretrained weights (bo fc layers)
ckpt_path = join(CHECKPOINT_DIR, 'netclr_final.pth')
checkpoint = torch.load(ckpt_path, map_location=device)

new_ckpt = {}
for k, v in checkpoint.items():
    if k.startswith('backbone.') and not k.startswith('backbone.fc'):
        new_ckpt[k[len('backbone.'):]] = v

log = model_ft.load_state_dict(new_ckpt, strict=False)
print(f'Loaded pretrained weights (missing: {log.missing_keys})')

# Prepare data
train_ft_dataset = TestData(x_train, y_train)
train_ft_loader = DataLoader(train_ft_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

test_dataset = TestData(x_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

ft_optimizer = torch.optim.Adam(model_ft.parameters(), lr=FINETUNE_LR)

# Training loop
best_acc = 0
best_state = None

print(f'Finetuning: {FINETUNE_EPOCHS} epochs, {len(train_ft_dataset)} train, {len(test_dataset)} test')

for epoch in range(FINETUNE_EPOCHS):
    model_ft.train()
    epoch_loss = []
    
    pbar = tqdm(train_ft_loader, desc=f'Finetune {epoch+1}/{FINETUNE_EPOCHS}', leave=False)
    for data, target in pbar:
        data = data.view(data.size(0), 1, data.size(1)).float().to(device)
        target = target.to(device)
        
        ft_optimizer.zero_grad()
        output = model_ft(data)
        loss = F.cross_entropy(output, target)
        loss.backward()
        ft_optimizer.step()
        
        epoch_loss.append(loss.item())
        pbar.set_postfix(loss=f'{loss.item():.4f}')
    
    # Test
    if (epoch + 1) % 5 == 0 or epoch == 0:
        model_ft.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for data, target in test_loader:
                data = data.view(data.size(0), 1, data.size(1)).float().to(device)
                target = target.to(device)
                output = model_ft(data)
                pred = output.argmax(dim=1)
                correct += pred.eq(target).sum().item()
                total += len(target)
        
        acc = correct / total
        if acc > best_acc:
            best_acc = acc
            best_state = model_ft.state_dict().copy()
        
        print(f'  Epoch {epoch+1} - loss: {np.mean(epoch_loss):.4f} - acc: {acc:.4f} - best: {best_acc:.4f}')

# Restore best
if best_state:
    model_ft.load_state_dict(best_state)

print(f'\nBest test accuracy: {best_acc:.4f}')

# =========================================================
# FINAL EVALUATION
# =========================================================
print('\n' + '='*60)
print('FINAL EVALUATION')
print('='*60)

model_ft.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for data, target in test_loader:
        data = data.view(data.size(0), 1, data.size(1)).float().to(device)
        output = model_ft(data)
        pred = output.argmax(dim=1).cpu().numpy()
        all_preds.extend(pred)
        all_targets.extend(target.numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

acc = accuracy_score(all_targets, all_preds)
prec = precision_score(all_targets, all_preds, average='weighted', zero_division=0)
rec = recall_score(all_targets, all_preds, average='weighted', zero_division=0)
clf_report = classification_report(all_targets, all_preds, target_names=classes, zero_division=0)

result_text = f"""=====================================
MODEL EVALUATION (NetCLR)
=====================================
Classes:   {num_classes}
Pretrain:  {PRETRAIN_EPOCHS} epochs, temp={TEMPERATURE}
Finetune:  {FINETUNE_EPOCHS} epochs
Accuracy:  {acc:.4f}
Precision: {prec:.4f}
Recall:    {rec:.4f}

Classification Report:
{clf_report}
"""

with open('netclr_results.txt', 'w') as f:
    f.write(result_text)
print(result_text)

# Per-class
print('Per-class accuracy:')
for c in range(num_classes):
    m = all_targets == c
    if m.sum() > 0:
        print(f'  {classes[c]:30s}: {np.mean(all_preds[m]==all_targets[m]):.2%} ({m.sum()} samples)')

print('\nDone!')



FINE-TUNING
Loaded pretrained weights (missing: ['fc.weight', 'fc.bias'])
Finetuning: 50 epochs, 222331 train, 24704 test


Finetune 1/50:   0%|          | 0/1736 [00:00<?, ?it/s]

  Epoch 1 - loss: 0.7432 - acc: 0.8929 - best: 0.8929


Finetune 2/50:   0%|          | 0/1736 [00:00<?, ?it/s]

Finetune 3/50:   0%|          | 0/1736 [00:00<?, ?it/s]

...

Finetune 47/50:   0%|          | 0/1736 [00:00<?, ?it/s]

Finetune 48/50:   0%|          | 0/1736 [00:00<?, ?it/s]

Finetune 49/50:   0%|          | 0/1736 [00:00<?, ?it/s]

Finetune 50/50:   0%|          | 0/1736 [00:00<?, ?it/s]

  Epoch 50 - loss: 0.0082 - acc: 0.9345 - best: 0.9366

Best test accuracy: 0.9366

FINAL EVALUATION
MODEL EVALUATION (NetCLR)
Classes:   500
Pretrain:  50 epochs, temp=0.5
Finetune:  50 epochs
Accuracy:  0.9345
Precision: 0.9376
Recall:    0.9345

Classification Report:
                                                                               precision    recall  f1-score   support

                                                                      24tv_ua       0.98      0.98      0.98        50
                                                                     2beeg_me       0.91      0.86      0.89        50
                                                                 3gpking_name       0.98      1.00      0.99        50
                                                             a1_bluesystem_me       1.00      0.92      0.96        50
                                                                  aagmaal_run       0.98      0.98      0.98        50
             